# 🐍 Curso de Python Básico: Aula 14 - Manipulação de Arquivos em Python (I/O)

**Bem-vindo à décima quarta aula!** Como seu professor de programação do CIAA-LPS, elaborei este notebook para apresentar os fundamentos da **Manipulação de Arquivos** (*Input/Output - I/O*) em Python.

Em sistemas operacionais e de telemetria, quase tudo envolve ler ou gravar informações em arquivos salvos no disco de armazenamento. No nosso laboratório de submarinos, precisamos gravar relatórios de missão, salvar históricos de manutenção física das embarcações, arquivar leituras de sensores e carregar arquivos de configuração. Dominar a leitura e escrita de arquivos de texto, JSON e CSV é essencial para construir qualquer aplicação real.

---

### 🎯 Objetivos da Aula
1. **Entender a Abertura de Arquivos**: Aprender sobre a função `open()` e seus diferentes modos (`'r'`, `'w'`, `'a'`).
2. **Utilizar o Gerenciador de Contexto**: Aprender a melhor prática de abertura com a estrutura `with`.
3. **Escrever em Arquivos**: Criar novos arquivos de texto e adicionar linhas com `write()` e `writelines()`.
4. **Ler Arquivos**: Utilizar métodos de leitura como `read()`, `readline()`, `readlines()` e loops de leitura eficientes.
5. **Manipular Arquivos e Diretórios**: Usar os módulos `os` e `pathlib` para verificar existência, renomear e excluir arquivos.
6. **Manipular JSON e CSV**: Ler e salvar dados usando os módulos nativos do Python `json` e `csv`.
7. **Exercícios Práticos**: Criar um diário de bordo e um gravador de logs estruturados de reator nuclear.


## 1. Abertura e Fechamento de Arquivos

Para trabalhar com um arquivo em Python, precisamos primeiro "abri-lo" com a função `open()`. A função `open()` recebe dois argumentos principais:
1. **Caminho do arquivo**: O nome ou caminho do arquivo no disco.
2. **Modo**: O tipo de operação que desejamos realizar.

### Modos Comuns de Abertura:
* `'r'` (Read - Padrão): Abre o arquivo para leitura. Se o arquivo não existir, gera um erro (`FileNotFoundError`).
* `'w'` (Write): Abre o arquivo para escrita. Se o arquivo já existir, ele será **sobrescrito** (apagando o conteúdo anterior). Se não existir, cria um novo.
* `'a'` (Append): Abre o arquivo para escrita no final (anexar). Se o arquivo existir, o novo texto é adicionado após o conteúdo antigo. Se não existir, cria um novo.

### O Gerenciador de Contexto `with` (Recomendado)

Antigamente, precisávamos chamar `f.close()` manualmente ao terminar de usar um arquivo. Se o programa travasse antes disso, o arquivo ficaria preso na memória, podendo ser corrompido.

> [!IMPORTANT]
> A melhor prática em Python é abrir os arquivos usando a cláusula **`with open(...) as f:`**. Isso cria um **Gerenciador de Contexto** que garante o fechamento automático do arquivo assim que o bloco de código indentado termina, mesmo se ocorrer um erro durante a execução.


In [ ]:
# Método Recomendado: Usando a estrutura 'with'
with open("/tmp/exemplo_arquivo.txt", "w", encoding="utf-8") as arquivo:
    # O arquivo está aberto apenas dentro deste bloco indentado
    arquivo.write("Registrando primeira linha de teste do CIAA-LPS.\n")

# Aqui fora, o arquivo já foi fechado automaticamente pelo Python!
print("Arquivo fechado com sucesso!")


## 2. Escrevendo em Arquivos

Temos duas funções principais para gravação de strings:
* `write(texto_string)`: Grava uma única string no arquivo.
* `writelines(lista_strings)`: Grava uma lista de strings sequencialmente no arquivo (não adiciona quebras de linha `\n` automaticamente; precisamos adicioná-las em cada string).


In [ ]:
caminho_diario = "/tmp/diario_bordo.txt"

# 1. Modo 'w' - Cria ou Sobrescreve
with open(caminho_diario, "w", encoding="utf-8") as f:
    f.write("--- DIÁRIO DE BORDO DO SUBMARINO CIAA ---\n")
    f.write("Status: Pronto para navegação.\n")

# 2. Modo 'a' - Anexa ao final do arquivo
with open(caminho_diario, "a", encoding="utf-8") as f:
    novos_logs = [
        "Evento: Motores iniciados com sucesso.\n",
        "Profundidade: 12 metros.\n"
    ]
    f.writelines(novos_logs)

print("Logs gravados no arquivo!")


## 3. Lendo Arquivos

Para ler as informações salvas no arquivo, abrimos em modo `'r'` e podemos usar diferentes métodos:
* `read()`: Lê todo o conteúdo do arquivo e o retorna como uma única string gigante.
* `readline()`: Lê apenas uma única linha do arquivo por chamada.
* `readlines()`: Lê todas as linhas do arquivo e as retorna dentro de uma **lista** de strings.
* **Iterador do arquivo** (Melhor Prática): Ler o arquivo em um loop `for linha in arquivo:`.


In [ ]:
print("--- 1. Lendo com f.read() (Tudo de uma vez) ---")
with open(caminho_diario, "r", encoding="utf-8") as f:
    conteudo = f.read()
    print(conteudo)

print("--- 2. Lendo com f.readlines() (Retorna uma Lista) ---")
with open(caminho_diario, "r", encoding="utf-8") as f:
    linhas = f.readlines()
    print("Lista de linhas:", linhas)

print("\n--- 3. Lendo Linha por Linha de forma eficiente (Loop for) ---")
# Recomendado para arquivos muito grandes, pois não carrega o arquivo inteiro na memória RAM
with open(caminho_diario, "r", encoding="utf-8") as f:
    for i, linha in enumerate(f, start=1):
        # Usamos .strip() para remover a quebra de linha \n extra no final de cada linha
        print(f"Linha {i}: {linha.strip()}")


## 4. Manipulação de Arquivos e Pastas (`os` e `pathlib`)

Para gerenciar os arquivos no disco (renomear, excluir, verificar se existem ou criar diretórios), usamos os módulos nativos `os` ou `pathlib` (mais moderno).


In [ ]:
import os
from pathlib import Path

caminho_arquivo = Path("/tmp/diario_bordo.txt")

# 1. Verificar se um arquivo existe antes de tentar lê-lo
if caminho_arquivo.exists():
    print(f"✅ O arquivo '{caminho_arquivo.name}' existe no disco!")
else:
    print("❌ Arquivo não encontrado.")

# 2. Criar uma nova pasta no disco de forma segura
pasta_logs = Path("/tmp/logs_ciaa_lps")
pasta_logs.mkdir(parents=True, exist_ok=True) # parents=True cria subpastas, exist_ok=True evita erro se já existir
print(f"Diretório '{pasta_logs}' criado ou verificado.")

# 3. Renomear arquivo
caminho_renomeado = Path("/tmp/diario_final.txt")
if caminho_arquivo.exists():
    os.rename(caminho_arquivo, caminho_renomeado)
    print(f"Arquivo renomeado para: {caminho_renomeado.name}")

# 4. Excluir arquivo (limpando nosso teste)
if caminho_renomeado.exists():
    os.remove(caminho_renomeado)
    print(f"Arquivo '{caminho_renomeado.name}' excluído do disco para limpeza.")


## 5. Manipulando Formatos Estruturados (JSON e CSV nativo)

### Manipulando Arquivos JSON
O formato JSON (JavaScript Object Notation) é amplamente utilizado para salvar arquivos de configurações em sistemas digitais. No Python, usamos o módulo `json`:
* `json.dump(objeto, arquivo)`: Serializa e grava um dicionário ou lista Python em um arquivo JSON.
* `json.load(arquivo)`: Lê e desserializa um arquivo JSON de volta em um dicionário Python.


In [ ]:
import json

configuracao_submarino = {
    "nome": "CIAA-LPS-Sub-1",
    "reator_ativo": True,
    "sensores_operacionais": ["Sonar-1", "Sonar-2", "Pressao-3"],
    "profundidade_limite": 400.0
}

caminho_json = "/tmp/config_submarino.json"

# Gravando Dicionário como arquivo JSON
with open(caminho_json, "w", encoding="utf-8") as f:
    json.dump(configuracao_submarino, f, indent=4, ensure_ascii=False)
print("Configurações salvas em formato JSON!")

# Lendo arquivo JSON de volta para dicionário Python
with open(caminho_json, "r", encoding="utf-8") as f:
    dados_carregados = json.load(f)
    print("\nDados Carregados do JSON:")
    print(dados_carregados)
    print("Tipo de dados carregado:", type(dados_carregados))


### Manipulando Arquivos CSV Nativo
Para ler e gravar tabelas simples em formato CSV sem precisar carregar a biblioteca Pandas, usamos o módulo nativo `csv`:


In [ ]:
import csv

dados_telemetria = [
    ["timestamp", "sensor", "valor"],
    ["10:00:00", "SENSOR-A", 82.5],
    ["10:01:00", "SENSOR-A", 83.1],
    ["10:02:00", "SENSOR-B", 12.4]
]

caminho_csv = "/tmp/telemetria.csv"

# Escrevendo no CSV nativamente
with open(caminho_csv, "w", newline="", encoding="utf-8") as f:
    escritor_csv = csv.writer(f)
    escritor_csv.writerows(dados_telemetria)
print("Dados salvos em formato CSV!")

# Lendo o CSV nativamente
with open(caminho_csv, "r", encoding="utf-8") as f:
    leitor_csv = csv.reader(f)
    print("\nLeitura do CSV Linha por Linha:")
    for linha in leitor_csv:
        print(linha)


## 6. Exercícios Práticos

---
### Exercício 1: Registrando o Diário de Bordo do Submarino

Para documentar as atividades diárias do submarino do CIAA-LPS, crie um script que atenda aos seguintes requisitos:

1. Abra um arquivo chamado `/tmp/diario_missao.txt` no modo de escrita (`'w'`).
2. Escreva as três linhas abaixo no arquivo usando `f.write()` (não se esqueça de adicionar a quebra de linha `\n` ao final de cada uma):
   - `"[08:00] Submarino desatracou da base naval do CIAA."`
   - `"[12:00] Realizada imersão de rotina a 50 metros de profundidade."`
   - `"[18:00] Todos os sistemas do reator operando normalmente."`
3. Em seguida, abra o arquivo no modo de leitura (`'r'`), leia o conteúdo linha por linha utilizando um loop `for`, e imprima cada linha precedida pelo seu número (ex: `"Linha 1: [08:00] ..."`).


In [ ]:
# Escreva aqui sua resolução para o Exercício 1
caminho_missao = "/tmp/diario_missao.txt"

# 1 e 2. Escrever no arquivo...


# 3. Ler e exibir com numeração de linhas...


---
### Exercício 2: Sistema de Logs Contínuos do Reator

Durante a navegação, precisamos registrar eventos e alertas no arquivo de log do reator. Como novos eventos acontecem continuamente, devemos usar o modo correto para não apagar os registros antigos.

1. Escreva uma função chamada `adicionar_log_reator(mensagem: str)` que abre o arquivo `/tmp/reator.log` em modo de **anexo (append)** e grava a mensagem seguida por uma nova linha.
2. Chame a função `adicionar_log_reator` três vezes com as seguintes mensagens de teste:
   - `"[ALERTA] Temperatura do Reator atingiu 275°C."`
   - `"[INFO] Acionado sistema de resfriamento secundário."`
   - `"[OK] Temperatura estabilizada em 255°C."`
3. Abra o arquivo `/tmp/reator.log` em modo de leitura e imprima todo o seu conteúdo na tela usando o método `read()` para verificar se todos os eventos foram gravados sequencialmente no mesmo arquivo.


In [ ]:
# Escreva aqui sua resolução para o Exercício 2
caminho_log_reator = "/tmp/reator.log"

# Limpando arquivo de testes anteriores
if os.path.exists(caminho_log_reator):
    os.remove(caminho_log_reator)

# 1. Definição da função adicionar_log_reator...
def adicionar_log_reator(mensagem: str):
    pass

# 2. Chamadas da função...


# 3. Leitura e exibição dos logs...


---

### 🎉 Parabéns!
Você concluiu a **Aula 14**! Agora você sabe lidar com arquivos de forma nativa e segura em Python, o que permitirá estruturar e automatizar a coleta e persistência de dados de simulação do laboratório CIAA-LPS!
